# Selective facts — corrected scope, existing weights
This is an **inference-only parser evaluation**. It reuses your selected full epoch-2 1.5B adapter. It does not train a model or run the six Stage 2 systems.

Choose **GPU T4 x2**, **Internet on**, and attach two private datasets:
1. The contents of **selective_facts_scopefix_source.zip** supplied with this notebook.
2. Your completed training output **selective_facts_1_5b.zip** (not the smoke ZIP). Kaggle can expose ZIPs as files or unpacked directories; both are supported.

No original adapter dataset, GitHub push, manual adapter path or earlier Stage 2 results are required. The loader verifies the exact selected weights and original SF1 metadata without changing them.

The scope correction prevents “Ignore missing values when computing statistics” from being interpreted as a row condition. It preserves the original user request and facts format. Preflight verifies that 12 previously blocked development cases now receive facts and the other 88 prompts remain exactly unchanged.

After a two-task inference smoke, it evaluates all 100 open-development cases freshly. `comparison.json` compares them with the historical original-adapter and SF1 results, with per-task gains/losses and the 12 blocked cases listed separately. Historical scores are comparisons, not reused current predictions. No selection/test inference is run. These development scores are not an independent benchmark or end-to-end Stage 2 accuracy.

Download **selective_facts_scopefix_eval.zip** and the executed notebook. Improved scores are not guaranteed; inspect the result before full Stage 2 runs.


In [ ]:
import hashlib,json,os,subprocess,sys,zipfile
from pathlib import Path
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
MANIFEST_SHA='21063deacc439e196c07bad84cf4b19ba25b99d2cf8f85a49587d457e7bb0e9b'
INPUT=Path('/kaggle/input')
MANIFEST_NAME='selective_facts_scopefix_manifest.json'
ROOT=WORK/('selective_facts_scopefix_source_'+MANIFEST_SHA[:12])

def write_source(manifest_bytes,read_source):
    assert hashlib.sha256(manifest_bytes).hexdigest()==MANIFEST_SHA, 'Wrong source/data bundle version'
    manifest=json.loads(manifest_bytes)
    assert manifest['experiment']=='selective-facts-scopefix1'
    for name,expected in manifest['sha256'].items():
        relative=Path(name)
        assert not relative.is_absolute() and '..' not in relative.parts, 'Invalid source path'
        content=read_source(name)
        assert hashlib.sha256(content).hexdigest()==expected, f'Changed source/data: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==content, f'Changed working copy: {name}'
        else:path.write_bytes(content)
    print('Verified source and data:',len(manifest['sha256']),'files',MANIFEST_SHA)

# Kaggle can expose uploaded ZIPs as files or unpacked dataset directories. Support both.
manifest_paths=[p for p in INPUT.rglob(MANIFEST_NAME) if hashlib.sha256(p.read_bytes()).hexdigest()==MANIFEST_SHA]
if manifest_paths:
    manifest_path=manifest_paths[0]
    write_source(manifest_path.read_bytes(),lambda name:(manifest_path.parent/name).read_bytes())
else:
    archives=list(INPUT.rglob('selective_facts_scopefix_source.zip'))
    found=False
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            assert len(z.namelist())==len(set(z.namelist())), 'Duplicate source ZIP members'
            name='selective_facts_scopefix_source/'+MANIFEST_NAME
            if name not in z.namelist():continue
            manifest_bytes=z.read(name)
            if hashlib.sha256(manifest_bytes).hexdigest()!=MANIFEST_SHA:continue
            write_source(manifest_bytes,lambda relative:z.read('selective_facts_scopefix_source/'+relative))
            found=True;break
    assert found, 'Attach the private dataset made from selective_facts_scopefix_source.zip, and the training output selective_facts_1_5b.zip.'
PACKAGES=WORK/'selective_facts_scopefix_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.__version__=='2.11.0+cu130'; assert torch.cuda.device_count()==2; assert all('T4' in torch.cuda.get_device_name(i) for i in (0,1)); print('T4 x2 preflight passed')"],cwd=ROOT,env=ENV,check=True)
# No optimizer or training is run. Check the one-visible-GPU inference topology.
subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES='0'),check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts_scopefix.preflight'],cwd=ROOT,env=ENV,check=True)
ADAPTER_PATH_FILE=WORK/'scopefix_adapter_path.txt'
subprocess.run([sys.executable,'-m','finetune.selective_facts_scopefix.adapter','--out',str(ADAPTER_PATH_FILE)],cwd=ROOT,env=ENV,check=True)
ADAPTER=Path(ADAPTER_PATH_FILE.read_text())
print('Existing epoch-2 adapter:',ADAPTER)


In [ ]:
def execute(smoke):
    out=WORK/('selective_facts_scopefix_smoke' if smoke else 'selective_facts_scopefix_eval')
    assert not out.exists(), 'Start a fresh session: existing results are never overwritten.'
    command=[sys.executable,'-m','finetune.selective_facts_scopefix.evaluate','--adapter',str(ADAPTER),'--out',str(out),'--packages',str(PACKAGES)]
    if smoke:command+=['--smoke']
    log=WORK/(out.name+'.log')
    with log.open('w') as f:
        process=subprocess.Popen(command,cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES='0'),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');f.write(line);f.flush()
        code=process.wait()
    assert code==0, f'Job failed. Download {log} and the partial ZIP for diagnosis.'
    assert json.loads((out/'completion.json').read_text())['complete']
    return out
SMOKE=execute(True)
print('Two-task inference smoke completed. The next cell evaluates all 100 development cases.')


In [ ]:
OUT=execute(False)
comparison=json.loads((OUT/'comparison.json').read_text())
print(json.dumps(comparison,indent=2))
from IPython.display import display,FileLink
display(FileLink(str(OUT.with_suffix('.zip'))))
print('Download selective_facts_scopefix_eval.zip and this executed notebook for analysis. No training was run.')
